# Neon-HTLC: evaluation

This notebook reproduces the evaluation of *Neon-HTLC: Collateral-Free HTLCs with Logarithmic Timeout* (ACM CCS 2026), i.e., Section 6 and Appendix C.

**How to use:** set the options in the *Configuration* cell below (most importantly `DISTRIBUTION`), then run all cells (*Kernel → Restart Kernel and Run All Cells*). CSV results are written to `results/<DISTRIBUTION>/data/` and figures to `results/<DISTRIBUTION>/figures/`, so runs with different distributions never overwrite each other. `README.md` lists the requirements (including LaTeX) and which cell produces which figure of the paper.

In [ ]:
import math
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import csv
import os
import scienceplots  # registers the 'science', 'ieee' and 'grid' styles; they render all text with LaTeX (see README.md)

In [ ]:
# ==========================================
# Configuration: the only cell you need to edit. Then run all cells.
# ==========================================

# Mining power distribution used by all experiments and figures (loaded from "<DISTRIBUTION>.txt"):
#   "real_distribution_69"            -> the simulated distribution used in the paper (Section 6, Table 4)
#   "real_distribution_519"           -> the longer tail distribution as presented in the appendix (Appendix C.4)
#   "real_distribution_decentralized" -> the decentralized distribution without real-world data (Appendix C.5)
DISTRIBUTION = "real_distribution_69"

# Evaluation parameters, the same for all protocols (Section 6)
FEE_A = 1.0    # Alice's transaction fee f_A in USD
T1 = 12        # payment-phase timeout T_1 in blocks
DEP_VALUES = [round(d, 2) for d in np.logspace(1, 5, num=500)]  # deposits from $10 to $100,000

# Optional: regenerate the distribution files with the two generator cells below. This is not needed to
# reproduce the paper, since the .txt files are included. If True, the generators overwrite
# real_distribution_<N>.txt and real_distribution_decentralized.txt. In both cases, the experiments
# use the distribution selected by DISTRIBUTION above.
REGENERATE_DISTRIBUTION_FILES = False
NUM_UNKNOWN_MINERS = 50  # small miners in the tail of the realistic generator: 50 -> 69 miners (paper), 500 -> 519 miners (appendix)

In this block below, we generate the mining power distribution used for our evaluation. We use the data from January 2026 in Bitcoin to fix the big mining pools, and simulate the remaining miners using a Zipf's law. This gives us a distribution close to the real world. We change the unknown miners amount to 500 to show that the small miners do not influence the comparison between protocol. 
Additionally, the decentralized distribution is made to show protocol performances under a more idealized decentralized bitcoin network, and we include the results in the appendix of our paper.

The two generator cells below only write the distribution files if `REGENERATE_DISTRIBUTION_FILES = True` in the configuration cell. They never change which distribution the experiments use: that is always the file selected by `DISTRIBUTION`, loaded in the cell after the generators.

In [ ]:
# Real-World Hash Power Distribution Generator (Based on Jan 1, 2026 Data). The result is only saved if REGENERATE_DISTRIBUTION_FILES = True.
# Set NUM_UNKNOWN_MINERS = 500 in the configuration cell to generate the longer tail distribution as presented in the appendix.

def generate_realistic_bitcoin_distribution(known_pools, unknown_total_hash, num_unknown_miners=500, exponent=1.5):
    """
    Combines known large pool data with a synthetically generated 'long tail' 
    of small miners following a Power Law (Zipf) distribution.
    """
    # 1. Generate the Zipf distribution for the tail
    tail_raw = [1.0 / (k ** exponent) for k in range(1, num_unknown_miners + 1)]
    
    # 2. Normalize the tail so its sum exactly equals the unknown_total_hash
    tail_sum = sum(tail_raw)
    tail_normalized = [(val / tail_sum) * unknown_total_hash for val in tail_raw]
    
    # 3. Combine known pools with the generated tail
    full_distribution = known_pools + tail_normalized
    
    # 4. Final normalization just to prevent floating point drift
    total_network = sum(full_distribution)
    
    # Return sorted (ascending) list 
    return sorted([p / total_network for p in full_distribution])

# --- Extracted Data (Converted to decimals) ---
# We exclude the 'Unknown' (1.64%) from this list, as we will generate it dynamically.
known_pools_jan_2026 = [
    0.2975, # Foundry USA
    0.1623, # AntPool
    0.1111, # ViaBTC
    0.1068, # F2Pool
    0.0822, # SpiderPool
    0.0524, # MARA Pool
    0.0465, # SecPool
    0.0284, # Binance Pool
    0.0260, # Luxor
    0.0186, # SBI Crypto
    0.0167, # OCEAN
    0.0153, # BraiinsPool
    0.0069, # WhitePool
    0.0036, # Ultimus Pool
    0.0029, # Poolin
    0.0029, # BTC.com
    0.0026, # MiningCore
    0.0007, # NiceHash
    0.0002  # KanoPool
]

unknown_hash_jan_2026 = 1 - sum(known_pools_jan_2026) # The 1.64% Unknown slice

# --- Generate the final distribution ---
# Assuming the 1.64% unknown hash power represents 50/500 tiny distinct entities/solo miners
# Change NUM_UNKNOWN_MINERS in the configuration cell to test different scenarios.
# The result is kept in its own variable, so it does not replace the distribution selected in the configuration.
generated_distribution = generate_realistic_bitcoin_distribution(
    known_pools=known_pools_jan_2026,
    unknown_total_hash=unknown_hash_jan_2026,
    num_unknown_miners=NUM_UNKNOWN_MINERS,
)

# --- Output Analysis ---
print(f"Total N (distinct mining entities): {len(generated_distribution)}")
print(f"Largest pool (Foundry): {generated_distribution[-1]*100:.2f}%")
print(f"Smallest solo miner (end of tail): {generated_distribution[0]*100:.8f}%")
print(f"All miners sorted (descending): {[f'{p*100:.4f}%' for p in sorted(generated_distribution, reverse=True)]}")
print(f"Total Distribution Sum: {sum(generated_distribution):.5f}")
if REGENERATE_DISTRIBUTION_FILES:
    # the file name follows the number of miners: 69 for the paper's distribution, 519 for the longer tail
    out_file = f"real_distribution_{len(generated_distribution)}.txt"
    np.savetxt(out_file, generated_distribution)
    print(f"Saved: {out_file}")

In [ ]:
# A more decentralized distribution generator without real-world data. The result is only saved if REGENERATE_DISTRIBUTION_FILES = True.

def generate_decentralized_distribution(num_miners=500, zipf_exponent=0.85):
    """
    Generates a realistic decentralized mining distribution using a flat power law.
    Returns a sorted array (ascending) summing to 1.0.
    """
    # 1. Generate the raw Zipf values: 1/1^s, 1/2^s, 1/3^s ... 1/N^s
    raw_dist = [1.0 / (k ** zipf_exponent) for k in range(1, num_miners + 1)]
    
    # 2. Normalize so the sum is exactly 1.0 (100% of hashpower)
    total_hash = sum(raw_dist)
    normalized_dist = [p / total_hash for p in raw_dist]
    
    # 3. Return sorted ascending 
    return sorted(normalized_dist)

# ==========================================
# Generate and verify the stats
# ==========================================
decentralized_distribution = generate_decentralized_distribution(num_miners=500, zipf_exponent=0.85)

print("--- Decentralized Network Stats ---")
print(f"Total distinct miners (N): {len(decentralized_distribution)}")
print(f"Largest single miner:      {decentralized_distribution[-1] * 100:.2f}%")
print(f"Top 5 miners combined:     {sum(decentralized_distribution   [-5:]) * 100:.2f}%")
print(f"Smallest solo miner:       {decentralized_distribution[0] * 100:.6f}%")
if REGENERATE_DISTRIBUTION_FILES:
    np.savetxt("real_distribution_decentralized.txt", decentralized_distribution)
    print("Saved: real_distribution_decentralized.txt")

In [ ]:
## Load the selected mining distribution and create the output directories of this distribution
real_distribution = np.loadtxt(f"{DISTRIBUTION}.txt")

OUTPUT_DIR = os.path.join("results", DISTRIBUTION)
DATA_DIR = os.path.join(OUTPUT_DIR, "data")      # CSV results of the experiments
FIG_DIR = os.path.join(OUTPUT_DIR, "figures")    # PDF figures
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

print(f"Distribution: {DISTRIBUTION} (N = {len(real_distribution)} miners)")
print(f"CSV results -> {DATA_DIR}/,  figures -> {FIG_DIR}/")

Below is a function to reproduce performances of FB-HTLC (The protocol presented by Nadahalli et. al.). They presented an approximate bound for the timeout. However, since their bounds only work for the final bribe, we do not consider their protocols in our final evaluation. Thus, this function is not used.

In [ ]:
# This code calculates the FB-HTLC timeout according to their approximated bound, which is not used for current experiments.
# ==========================================

def calculate_approx_T(dep, feeA, miners):
    """
    Calculates the approximate required Timeout (T) using the formula:
    T > log_{1 - p_w}(pun/dep)
    """
    pun = feeA
    p_w = sum(p for p in miners if p < (pun / dep))
    
    if p_w == 0:
        return float('inf') 
    if p_w == 1 or p_w >= (1 - 1e-9):
        return 1 
    try:
        T_approx = math.log(pun / dep) / math.log(1 - p_w)
        return math.floor(T_approx) + 1
    except ValueError:
        return 1

Next we define the function to calculate the expected number of distinct miners given a distribution and a timeout T. This function is used for all two-phase protocols we evaluated and for the tight bound of Neon-HTLC.
We also plot the relationship between timeout and kappa to show the trend.

In [ ]:
#This function returns the expected number of distinct miners given a distribution and a timeout T. 
def expected_distinct_miners(miners, T):
    """
    Calculates kappa: the expected number of distinct miners 
    who will mine at least one block over a period of T blocks.
    Rounded up to the nearest whole integer to strictly preserve security.
    We use ceil here but floor+1 for T since T needs to be strictly greater but kappa can be equal.
    """
    N = len(miners)
    # Formula: N - sum((1 - p_i)^T)
    kappa = N - sum((1 - p) ** T for p in miners)
    return math.ceil(kappa)

# expected_distinct_miners(real_distribution, 20)

In [ ]:
# Paper: Figure 9 (real_distribution_69), Figure 10 (real_distribution_519), Figure 14 (real_distribution_decentralized).
# this function plots the growth of expected distinct miners (kappa) as a function of timeout T, using the provided miner distribution. 
def plot_kappa_trend_linear(miners, max_blocks=5000):
    N = len(miners)
    plt.style.use(['science', 'ieee', 'grid'])
    T_values = list(range(1, max_blocks + 1))
    kappa_values = [expected_distinct_miners(miners, T) for T in T_values]

    plt.figure(figsize=(6,3.6 ), dpi=120)
    
    # Plot the curve
    plt.plot(T_values, kappa_values, color='#1f77b4', linewidth=2.5)
    
    # Add Asymptote
    plt.axhline(y=N, color='#d62728', linestyle='--', linewidth=2, 
                label=f'Absolute Maximum (N={N})')

    plt.xlabel('Time Interval in Blocks', fontsize=12)
    plt.ylabel(r'$\kappa$', fontsize=12)
    # plt.ylim(0, 70) 

    
    plt.xticks([1, 300, 600, 1000, 2000, 5000], ['1', '300', '600', '1,000', '2,000', '5,000'])
    plt.tick_params(axis='x', which='major', labelsize=9)
    
    # Grids
    plt.grid(True, which='major', linestyle='-', alpha=0.5)
    plt.grid(True, which='minor', linestyle=':', alpha=0.2)
    
    plt.legend(loc='lower right', fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, 'kappa-linear.pdf'), format='pdf', bbox_inches='tight')
    plt.show()

plot_kappa_trend_linear(real_distribution, max_blocks=5000)

Below is the core functions for us to evaluate the protocols. We implement several functions accoding to the protocol description, to calculate their required safe timeouts given a set of parameters.

In [ ]:
## Neon-HTLC Tight bound
def expected_escalating_bribe_cost(miners, Delta, pun, dep, col):
    """
    Calculates the exact expected total cost for Bob to bribe the network
    during an escalating punishment phase of length Delta.
    """
    total_expected_cost = 0.0
    
    for p_i in miners:
        expected_payout_for_miner_i = 0.0
        
        # Calculate expected payout for this specific miner
        for t in range(1, Delta + 1):
            # Probability that block t is the LAST block this miner finds
            prob_highest_block_is_t = p_i * ((1 - p_i) ** (Delta - t))
            
            # The payout the attacker must match if block t is the highest
            theoretical_payout = (2 ** (t - 1)) * pun
            actual_payout_t = min(theoretical_payout, dep+col)
            
            expected_payout_for_miner_i += prob_highest_block_is_t * actual_payout_t
            
        total_expected_cost += expected_payout_for_miner_i
        
    return total_expected_cost

def calculate_T_bribe_escalating_tight(dep, pun, miners, col, fee_A, T1):
    """
    Finds the minimum Delta (T2) using the exact expected statistical cost.
    """       
    Delta = 1
    kappa_A = expected_distinct_miners(miners, T1)
    while True:
        # Get the exact expected cost for this Delta
        required_total_bribe = expected_escalating_bribe_cost(miners, Delta, pun, dep, col)
        required_total_bribe = round(required_total_bribe, 6) 
        
        if required_total_bribe >= dep - kappa_A * fee_A:
            return Delta
            
        Delta += 1

In [ ]:
##Main function for all two-phase calculations. We use flags to determine which specific protocol's logic to apply. 
def calculate_two_phase_pun(miners, T1, fee_A, col, d1=True):
    """
    Calculates the punishment (pun) derived from the reverse bribery bound.
    """
    kappa_A = expected_distinct_miners(miners, T1)
    censA = kappa_A * fee_A
    if d1: # If we wait delta blocks after T1 to break the success-dependency of RB attack. This is always used whenever we calculate punishment from the RB bound.
        pun = 2 * (col + censA)
    else: #edge case that is not used in the paper or the evaluation. This gives no advantage.
        pun = col + censA
    return pun

def calculate_T_bribe_kappa(dep, pun, fee_A, T1, miners, col, he=False):
    """
    Finds the minimum T2 for a protocol to be safe (incentive-compatible).
    """
    N = len(miners)
    kappa_A = expected_distinct_miners(miners, T1)
    
    # Edge Case Check: If the attacker can afford to bribe EVERY miner, 
    # no T will ever be safe for the two-phase protocols
    if N * pun <= dep:
        # print(f"Warning: dep ({dep}) is large enough to bribe all {N} miners (cost: {N * pun}).")
        return float('inf') 
    
    if he:
        # for HE-HTLC, we want to calculate the kappa from the given col, dep and feeA according to their relationship: col >= dep/(kappa-1) + feeA
        # rearranging gives us kappa >= (dep/(col-feeA)) + 1
        # Since He-HTLC demands kappa >=2, this gives us a natural lower bound for collateral whenever a dep is set. If we want to try 5%, we need to start at dep=30.
        if col <= fee_A:
            return float('inf') # Impossible case, collateral must be greater than feeA to have any valid kappa.
        kappa_target = math.ceil((dep / (col - fee_A)) + 1)
        # print(f"HE-HTLC Case: Calculated kappa based on col and dep is {kappa_target}.")
        if kappa_target > N or kappa_target < 2:
            print(f"Warning: Calculated kappa target ({kappa_target}) exceeds total number of miners ({N}).")
            return float('inf')
    
    T = 1
    while True:
        kappa = expected_distinct_miners(miners, T)
        if he:
            required_total_bribe = -100 # to prevent accidentally exiting the loop.
            if kappa >= kappa_target:
                # print(f"HE-HTLC Case: At T={T}, kappa={kappa} meets the required kappa_target={kappa_target}.")
                return T
        else:
            required_total_bribe = kappa * pun

        # If the required bribe exceeds the deposit, the attack is unfundable
        if required_total_bribe >= dep - kappa_A * fee_A: ## - bribeA?
            # print(f"HE-HTLC Case: At T={T}, kappa={kappa} meets the required kappa_target={kappa_target}.")
            # print(f"Bribery now requires {required_total_bribe:.2f}.")
            # print(f"kappa: {kappa}")
            return T
            
        T += 1

def calculate_two_phase_T(dep, fee_A, col, T1, miners, d1=False, he=False, neon_tight=False):
    """
    Master function to calculate the total safe Timeout (T) for the Two-Phase Protocol and Neon-HTLC tight bound.
    Total T = T1 + T2 (or 1 + T1 + T2 if d1 is True).
    """
    # 1. Define the punishment
    if he:
        pun = col # For HE-HTLC, the punishment is just the collateral
    else:
        pun = calculate_two_phase_pun(miners, T1, fee_A, col, d1) 
    
    # 2. Calculate Phase 2 required timeout (T2)
    
    if neon_tight:
        ### We use a different entry to calculate the neon-htlc tight bound, and also a completely different function to calculate the agnostic bound for neon-htlc, outside of this function.
        T2 = calculate_T_bribe_escalating_tight(dep, pun, miners, col, fee_A, T1)
    else:
        T2 = calculate_T_bribe_kappa(dep, pun, fee_A, T1, miners, col, he)
    # If the attacker's budget is so huge they can bypass Phase 2 entirely
    if T2 == float('inf'):
        return float('inf')
        
    # 3. Sum the total T
    total_T = T1 + T2
    if d1:
        total_T += 1
        
    return total_T


# ### Playground for testing the functions.
# dep = 12
# frac = 0.10
# fee = 1
# col = dep * frac
# calculate_two_phase_T(dep, fee, col, 12, real_distribution, d1=False, he=True)
# # calculate_two_phase_T(dep, fee, 0, 12, real_distribution, d1=True, he=False, neon_tight=True)

In [ ]:
## Neon-Agnostic. Since this evaluation is easy and independent of the distribution, we put it in a separate function. 
def calculate_two_phase_T_agnostic(dep, fee_A, col, T1, d1=True):
    """
    Calculates the completely Distribution-Agnostic safe timeout.
    Assumes the absolute worst-case scenario (kappa = 1).
    """
    # 1. Agnostic Phase 1 Punishment (Assume kappa_A = 1)
    if d1:
        pun_min = 2 * (col + fee_A)
    else:
        pun_min = col + fee_A
        
    if pun_min <= 0:
        return float('inf')
        
    # 2. Agnostic Phase 2 Timeout
    T2 = 1
    while True:
        # Attacker's absolute minimum possible cost
        theoretical_payout = (2**(T2 - 1)) * pun_min
        actual_payout = min(theoretical_payout, dep+col)
        
        if actual_payout >= dep - fee_A: ## - bribeA?
            break
        T2 += 1
        
    # 3. Sum the total T
    total_T = T1 + T2
    if d1:
        total_T += 1
        
    return total_T
    

In [ ]:
## Generate files for Neon-HTLC with agnostic bounds. We generate the results for analysis and figures.
dep_values = DEP_VALUES
## the fee and T1 are set in the configuration cell (FEE_A, T1) according to our assumption for the evaluation. This holds for all protocols.
fee_A = FEE_A

# 3. Define the Agnostic files to generate
agnostic_tests = [
    {'name': 'Neon_Agnostic_col0', 'col_flat': 0.0},
    {'name': 'Neon_Agnostic_col5pct', 'col_ratio': 0.05},
    {'name': 'Neon_Agnostic_col50_flat', 'col_flat': 50.0},
]

# 4. Generate and Save!
os.makedirs(DATA_DIR, exist_ok=True)

print("Generating Agnostic Bounds...")
for test in agnostic_tests:
    filepath = os.path.join(DATA_DIR, f"{test['name']}.csv")
    results = []
    
    for dep in dep_values:
        # Determine collateral based on the test type
        if 'col_ratio' in test:
            # col = (dep * test['col_ratio']) + fee_A
            col = (dep * test['col_ratio'])
        else:
            col = test['col_flat']
            
        # Calculate
        T_result = calculate_two_phase_T_agnostic(dep, fee_A, col, T1, d1=True)
        
        # Save
        str_T = "inf" if T_result == float('inf') else str(T_result)
        results.append({"Deposit_USD": dep, "Safe_T": str_T})
        
    # Write CSV
    with open(filepath, mode='w', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=["Deposit_USD", "Safe_T"])
        writer.writeheader()
        writer.writerows(results)
        
    print(f"Saved: {filepath}")

In [ ]:
### Paper: Figure 8 (Appendix C.2).
### This code is particularly for evaluating he-htlc, which we present in our appendix. It plots the relationship between Collateral and Timeout for HE-HTLC given a fixed deposit. 
### We show this because He-HTLC doesn't make it clear in their paper about how kappa influences the timeout (\ell in their paper).

def plot_collateral_vs_timeout(fixed_dep, fee_A, T1, miners):
    """
    Plots the relationship between Collateral and Timeout.
    """
    N = len(miners)
    plt.style.use(['science', 'ieee', 'grid']) 

    # 1. Calculate the Theoretical Minimum Collateral
    col_min_theoretical = (fixed_dep / (N - 1)) + fee_A
    
    print(f"--- Analysis for Deposit: \\${fixed_dep:,.0f} ---")
    print(f"Total Miners (N): {N}")
    print(f"Absolute Minimum Collateral Required: \\${col_min_theoretical:,.2f}")
    
    # 2. Generate a smart X-axis (Collateral range)
    # We start just slightly above the theoretical minimum (e.g., +1%) 
    # to avoid plotting 'infinity', and sweep up to 35% of the deposit.
    start_col = col_min_theoretical + 10
    end_col = fixed_dep * 0.35
    col_range = np.linspace(start_col, end_col, num=3000)
    
    T_values = []
    
    # 3. Calculate safe T for each collateral value
    for col in col_range:
        # Using the standard Two-Phase function (escalating=False, d1=False)
        # Since we only want to show T2 here, we subtract T1 from the total T to isolate the Phase 2 timeout.
        # In He-HTLC, T1 is irrelevant to the phase 2 completely.
        T = calculate_two_phase_T(fixed_dep, fee_A, col, T1, miners, d1=False, he=True) - T1
        T_values.append(T)
        
    # --- 4. Plotting ---
    plt.figure(figsize=(6, 4.5), dpi=120)
    
    # The main curve
    plt.plot(col_range, T_values, color='#2ca02c', linewidth=2.5)
    
    # The Vertical Asymptote (Theoretical Col_Min)
    plt.axvline(x=col_min_theoretical, color='red', linestyle=':', linewidth=2,
                label=f'Theoretical Col Min (\\${col_min_theoretical:,.0f})')

    plt.ylim(0,400)
    ticks = list(range(0, 101, 20)) + list(range(100, 401, 50))
    plt.yticks(ticks)

    # Formatting
    plt.xlabel('Collateral Amount in USD ($v^{col}$)', fontsize=12)
    plt.ylabel('Required Safe Timeout in Blocks ($T_2$)', fontsize=12)
    
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend(loc='upper right', fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, 'he-htlc.pdf'), format='pdf', bbox_inches='tight')    
    plt.show()

# ==========================================
# How to run it:
# ==========================================
plot_collateral_vs_timeout(fixed_dep=10000, fee_A=FEE_A, T1=T1, miners=real_distribution)

In [ ]:
# This function runs the full set of experiments (except for Neon-HTLC agnostic bound, which is already generated above) for the protocols and saves the results to CSV files.
def run_experiments(global_params, protocols_list, dep_values):
    """
    Loops through the requested protocols and saves each to its own CSV file.
    """
    os.makedirs(DATA_DIR, exist_ok=True)
    fee_A = global_params['fee_A']
    miners = global_params['miners']
    T1 = global_params['T1']
    
    for protocol in protocols_list:
        name = protocol['name']
        p_type = protocol['type']
        filepath = os.path.join(DATA_DIR, f"{name}.csv")
        
        print(f"Running '{name}'...")
        results = []
        
        for dep in dep_values:
            T_result = None
            
            # --- Switchboard ---
            if p_type == 'approx': # For FB-HTLC approximation, not used in current experiments.
                T_result = calculate_approx_T(dep, fee_A, miners)
            elif p_type == 'two_phase':
                # Handle dynamic collateral ratio OR flat collateral
                if 'col_ratio' in protocol:
                    # col = (dep * protocol['col_ratio']) + fee_A 
                    col = (dep * protocol['col_ratio']) 
                else:
                    col = protocol.get('col', 0.0) # Default to 0 if not provided
                
                # Extract boolean flags (defaulting to False if not specified)
                d1 = protocol.get('d1', False)
                he = protocol.get('he', False)
                neon_tight = protocol.get('neon_tight', False)
                
                T_result = calculate_two_phase_T(
                    dep, fee_A, col, T1, miners, 
                    d1=d1, he=he, neon_tight=neon_tight
                )
            else:
                raise ValueError(f"Unknown protocol type: {p_type}")
                
            # Format infinity for CSV saving
            str_T = "inf" if T_result == float('inf') else str(T_result)
            results.append({"Deposit_USD": dep, "Safe_T": str_T})
            # early stopping for infinite T to save time and resources. It doesn't work for col-ratio because T is decreasing from infinity.
            if T_result == float('inf') and not 'col_ratio' in protocol:
                print(f" Reached infinite T at deposit {dep}. Stopping further calculations for this protocol.")
                break

            
        # Write to dedicated CSV
        with open(filepath, mode='w', newline='') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=["Deposit_USD", "Safe_T"])
            writer.writeheader()
            writer.writerows(results)
            
    print(f"\n All requested experiments generated and saved to the '{DATA_DIR}/' folder.")

# ==========================================
# 4. Execution Dashboard
# ==========================================
if __name__ == "__main__":
    
    # 1. Global Parameters
    GLOBAL_PARAMS = {
        'fee_A': FEE_A,               
        'T1': T1,           
        'miners': real_distribution # load distribution earlier
    }
    
    # 2. Deposit Array ($10 to $100,000), set in the configuration cell
    dep_values = DEP_VALUES
    
    # 3. Master Protocol List
    PROTOCOLS_TO_RUN = [
        # # --- Baselines ---
        # {'name': 'IC_HTLC_Approx', 'type': 'approx'},
        
        # # # --- He-HTLC (Using col_ratio to match playground) ---
        {'name': 'He_HTLC_col5pct',  'type': 'two_phase', 'col_ratio': 0.05, 'he': True},
        {'name': 'He_HTLC_col10pct', 'type': 'two_phase', 'col_ratio': 0.10, 'he': True},
        {'name': 'He_HTLC_col20pct', 'type': 'two_phase', 'col_ratio': 0.20, 'he': True},

        # --- He-HTLC with Flat Collateral ---
        {'name': 'He_HTLC_col50_flat', 'type': 'two_phase', 'col': 50.0, 'he': True},
        {'name': 'He_HTLC_col20_flat', 'type': 'two_phase', 'col': 20.0, 'he': True},        

        # # --- He-plus (No Escalation) ---
        {'name': 'He_plus_col0',     'type': 'two_phase', 'col': 0.0, 'd1': True},
        {'name': 'He_plus_col20',    'type': 'two_phase', 'col': 20.0, 'd1': True},
        {'name': 'He_plus_col50',    'type': 'two_phase', 'col': 50.0, 'd1': True},
        {'name': 'He_plus_col5pct', 'type': 'two_phase', 'col_ratio': 0.05, 'd1': True},
        {'name': 'He_plus_col10pct', 'type': 'two_phase', 'col_ratio': 0.10, 'd1': True},
        {'name': 'He_plus_col20pct', 'type': 'two_phase', 'col_ratio': 0.20, 'd1': True},

        
        # # --- Neon (Tight Expectation) ---
        {'name': 'Neon_Tight_col0',    'type': 'two_phase', 'col': 0.0, 'd1': True, 'neon_tight': True},
        {'name': 'Neon_Tight_col5pct', 'type': 'two_phase', 'col_ratio': 0.05, 'd1': True, 'neon_tight': True},
        # {'name': 'Neon_Tight_col10pct', 'type': 'two_phase', 'col_ratio': 0.10, 'd1': True, 'neon_tight': True},

        {'name': 'Neon_Tight_col50_flat', 'type': 'two_phase', 'col': 50.0, 'd1': True, 'neon_tight': True},
    
        
    ]
    
    # Run all generations
    run_experiments(GLOBAL_PARAMS, PROTOCOLS_TO_RUN, dep_values)

Below are functions that use the data we just generated to draw the figures of the paper (Figures 5–7 for `real_distribution_69`). For the corresponding appendix figures, select another `DISTRIBUTION` in the configuration cell and run all cells again; `README.md` lists all figures.

In [ ]:
# Paper: Figure 7 (Figure 12 for real_distribution_519, Figure 16 for real_distribution_decentralized).
# This function plots the comparison graph for the Neon-HTLC protocol with different bounds (agnostic and tight), which is one of the main figures used in the evaluation.
def plot_escalation_comparison(plot_configs, y_max=50):
    fig, ax = plt.subplots(dpi=150)
    plt.style.use(['science', 'ieee', 'grid'])
    
    
    for config in plot_configs:
        filepath = config['filepath']
        label = config['label']
        color = config.get('color', '#000000')
        linestyle = config.get('linestyle', '-')
        
        if not os.path.exists(filepath):
            print(f" Missing: {filepath}")
            continue
            
        x_deps, y_timeouts = [], []
        
        with open(filepath, mode='r') as csvfile:
            reader = csv.DictReader(csvfile)
            for row in reader:
                dep = float(row['Deposit_USD'])
                t_val = row['Safe_T']
                if t_val != 'inf':
                    x_deps.append(dep)
                    y_timeouts.append(float(t_val))
                        
        ax.plot(x_deps, y_timeouts, label=label, color=color, linestyle=linestyle)

    # ==========================================
    # Formatting
    # ==========================================
    ax.set_ylim(12, 33)
    ax.set_xscale('log')
    ax.set_xlim(10, 100000)
    
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f"\\${int(x):,}"))
    
    ax.set_xlabel(r'Deposit Amount (USD) - Logarithmic Scale')
    ax.set_ylabel(r'Required Safe Timeout (blocks)')

    ax.grid(True, which='major', linestyle='-', alpha=0.3)
    ax.grid(True, which='minor', linestyle=':', alpha=0.1)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.22), ncol=3, framealpha=0.9, fontsize=7, columnspacing=0.7, handletextpad=0.5, handlelength=1.5)
    
    plt.savefig(os.path.join(FIG_DIR, 'Neon_Escalation.pdf'), format='pdf', bbox_inches='tight')
    plt.show()

# ==========================================
# The 9-Protocol Configuration Matrix
# ==========================================
if __name__ == "__main__":
    
    escalation_configs = [
        # --- 0 Collateral (BLUES) ---
        {'filepath': os.path.join(DATA_DIR, 'Neon_Tight_col0.csv'),    'label': r'Tight (0 Col)',    'color': '#1f77b4', 'linestyle': '-'},
        {'filepath': os.path.join(DATA_DIR, 'Neon_Agnostic_col0.csv'),     'label': r'Agnostic (0 Col)', 'color': '#1f77b4', 'linestyle': ':'},
        
        # --- Flat $50 Collateral (REDS) ---
        {'filepath': os.path.join(DATA_DIR, 'Neon_Tight_col50_flat.csv'),'label': r'Tight (Flat \$50 Col)',   'color': '#d62728', 'linestyle': '-'},
        {'filepath': os.path.join(DATA_DIR, 'Neon_Agnostic_col50_flat.csv'), 'label': r'Agnostic (Flat \$50 Col)','color': '#d62728', 'linestyle': ':'},

        # --- 5% Collateral (GREENS) ---
        {'filepath': os.path.join(DATA_DIR, 'Neon_Tight_col5pct.csv'), 'label': r'Tight (5\% Col)',  'color': '#2ca02c', 'linestyle': '-'},
        {'filepath': os.path.join(DATA_DIR, 'Neon_Agnostic_col5pct.csv'),  'label': r'Agnostic (5\% Col)','color': '#2ca02c', 'linestyle': ':'},

    ]
    
    plot_escalation_comparison(escalation_configs, y_max=50)

In [ ]:
# Paper: Figure 5 (Figure 11 for real_distribution_519, Figure 15 for real_distribution_decentralized).
# This function plots the comparison graph for the two-phase protocols with fixed collaterals or zero collateral, which is one of the main figures used in the evaluation.
def plot_legacy_failures_linear(plot_configs, y_max=600, x_max=4000):
    plt.style.use(['science', 'ieee', 'grid'])
    
    fig, ax = plt.subplots(figsize=(3.8, 4.2), dpi=150) 
    
    for config in plot_configs:
        filepath = config['filepath']
        if not os.path.exists(filepath): continue
            
        x_deps, y_timeouts = [], []
        
        with open(filepath, mode='r') as csvfile:
            for row in csv.DictReader(csvfile):
                dep, t_val = float(row['Deposit_USD']), row['Safe_T']
                
                # If we exceed our X-axis limit, stop plotting this line
                if dep > x_max:
                    break
                    
                if t_val == 'inf':
                    x_deps.append(dep); y_timeouts.append(y_max * 5); break 
                else:
                    t_float = float(t_val)
                    x_deps.append(dep); y_timeouts.append(t_float)
                    if t_float > y_max * 5: break
                        
        ax.plot(x_deps, y_timeouts, label=config['label'], color=config.get('color', '#000'), 
                linestyle=config.get('linestyle', '-'), linewidth=1.6)
        # Draw the theoretical vertical asymptote if it exists
        if 'limit' in config:
            ax.axvline(x=config['limit'], color=config.get('color'), 
                       linestyle=':', linewidth=1.4, alpha=0.6, label=config.get('limit_label', None))

    # ==========================================
    # Formatting (LINEAR X-AXIS)
    # ==========================================
    ax.set_ylim(0, y_max)
    
    ax.set_xlim(0, x_max) 
    
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f"\\${int(x):,}"))
    
    ax.set_xlabel(r'Deposit Amount (USD)')
    ax.set_ylabel(r'Required Safe Timeout (blocks)')
    
    #"Impractical Timeout" Red Zone at the top
    ax.axhspan(y_max * 0.95, y_max, color='red', alpha=0.15)
    
    ax.text(x=x_max * 0.52, y=y_max * 0.960, s=r'\textit{Impractical Timeout - 1 month}', color='darkred', fontsize=7)

    ax.legend(loc='lower right', bbox_to_anchor=(0.98, 0.01), framealpha=0.9, edgecolor='black', fancybox=False, fontsize=7, handletextpad=0.5)
    
    plt.savefig(os.path.join(FIG_DIR, 'HTLC_Legacy_Dep_Time.pdf'), format='pdf', bbox_inches='tight')
    plt.show()

if __name__ == "__main__":
    legacy_configs = [
        {'filepath': os.path.join(DATA_DIR, 'He_plus_col0.csv'),    'label': r'He$^+$-HTLC (0 Col)', 'color': '#d62728', 'linestyle': '-', 'limit': 973, 'limit_label': r'He$^+$ limit (0 Col)'},
        {'filepath': os.path.join(DATA_DIR, 'He_HTLC_col20_flat.csv'), 'label': r'He-HTLC (Flat \$20 Col)',          'color': '#2ca02c', 'linestyle': '-.', 'limit': 1292, 'limit_label': r'He limit (Flat \$20 Col)'},
        {'filepath': os.path.join(DATA_DIR, 'He_plus_col20.csv'),   'label': r'He$^+$-HTLC (Flat \$20 Col)',     'color': '#2ca02c', 'linestyle': '-', 'limit': 3733, 'limit_label': r'He$^+$ limit (Flat \$20 Col)'},
        {'filepath': os.path.join(DATA_DIR, 'He_HTLC_col50_flat.csv'), 'label': r'He-HTLC (Flat \$50 Col)',          'color': '#1f77b4', 'linestyle': '-.', 'limit': 3332, 'limit_label': r'He limit (Flat \$50 Col)'},
        {'filepath': os.path.join(DATA_DIR, 'He_plus_col50.csv'),   'label': r'He$^+$-HTLC (Flat \$50 Col)',     'color': '#1f77b4', 'linestyle': '-', 'limit': 7873, 'limit_label': r'He$^+$ limit (Flat \$50 Col)'}
    ]
    
    # The 'limit' values above are the payment caps (vertical asymptotes, Eq. (7)) of real_distribution_69
    # (N = 69, kappa(T1) rounded up = 7): He-HTLC (N-1)(v^col - f_A), He+-HTLC 2N v^col + (2N+1) f_A kappa_1.
    # They do not hold for other distributions, so they are only drawn for real_distribution_69
    # (the appendix figures for the other distributions show no asymptotes either).
    if DISTRIBUTION != "real_distribution_69":
        for config in legacy_configs:
            config.pop('limit', None)
            config.pop('limit_label', None)

    plot_legacy_failures_linear(legacy_configs, y_max=4400, x_max=8000)

In [ ]:
## Paper: Figure 6 (Figure 13 for real_distribution_519, Figure 17 for real_distribution_decentralized).
## This function plots the protocols with a collateral percentage against the deposit, which is one of the main figures used in the evaluation.
## To show the trend clearly, we also add an inset zoom box that focuses on the small timeout region.

from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

def plot_percentage_collaterals_inset(plot_configs, y_max=400):
    plt.style.use(['science', 'ieee', 'grid'])
    fig, ax = plt.subplots(figsize=(4.5, 3.0), dpi=150) 
    
    axins = ax.inset_axes([0.42, 0.35, 0.56, 0.6])
    
    for config in plot_configs:
        filepath = config['filepath']
        if not os.path.exists(filepath): continue
            
        x_deps, y_timeouts = [], []
        
        with open(filepath, mode='r') as csvfile:
            for row in csv.DictReader(csvfile):
                dep, t_val = float(row['Deposit_USD']), row['Safe_T']
                if t_val == 'inf':
                    x_deps.append(dep); y_timeouts.append(y_max * 5); continue
                else:
                    t_float = float(t_val)
                    x_deps.append(dep); y_timeouts.append(t_float)
                    # if t_float > y_max * 5: break
                        
        ax.plot(x_deps, y_timeouts, label=config['label'], color=config.get('color', '#000'), 
                linestyle=config.get('linestyle', '-'), linewidth=1.5)
        
        axins.plot(x_deps, y_timeouts, color=config.get('color', '#000'), 
                   linestyle=config.get('linestyle', '-'), linewidth=1.5)

    ax.set_ylim(0, y_max)
    ax.set_xlim(10, 100000)
    ax.set_xscale('log')
    ax.set_xticks([10, 20, 100, 1000, 10000, 100000])
    
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: f"\\${int(x):,}"))
    ax.set_xlabel(r'Deposit Amount (USD) - Logarithmic Scale')
    ax.set_ylabel(r'Required SafeTimeout (blocks)')
    
    ax.grid(True, which='major', linestyle='-', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.18), 
            framealpha=0.9, ncol=3, fontsize=7)
    

    axins.set_xlim(10, 5000) 
    axins.set_xscale('log')
    axins.set_ylim(12, 80)     
    
    axins.grid(True, linestyle=':', alpha=0.5)
    axins.set_yticks([ 12, 20, 30, 40, 50, 60])
    axins.set_yticklabels([12, 20, 30, 40, 50, 60], fontsize=7)
    axins.set_xticks([ 20, 100, 1000])
    
    axins.set_xticklabels([r'\$20', r'\$100', r'\$1000'], fontsize=7) 
    
    mark_inset(ax, axins, loc1=3, loc2=4, fc="none", ec="0.5", linestyle=':')
    
    plt.savefig(os.path.join(FIG_DIR, 'HTLC_Percentage_Collaterals.pdf'), format='pdf', bbox_inches='tight')
    plt.show()

if __name__ == "__main__":
    pct_configs = [
        {'filepath': os.path.join(DATA_DIR, 'He_HTLC_col5pct.csv'),  'label': r'He-HTLC (5\% Col)',  'color': '#d62728', 'linestyle': '-'},
        {'filepath': os.path.join(DATA_DIR, 'He_plus_col5pct.csv'), 'label': r'He$^+$-HTLC (5\% Col)', 'color': '#d62728', 'linestyle': '--'},
        {'filepath': os.path.join(DATA_DIR, 'He_HTLC_col10pct.csv'), 'label': r'He-HTLC (10\% Col)', 'color': '#ff7f0e', 'linestyle': '-'},
        {'filepath': os.path.join(DATA_DIR, 'He_HTLC_col20pct.csv'), 'label': r'He-HTLC (20\% Col)', 'color': "#6893e2", 'linestyle': '-'},
        {'filepath': os.path.join(DATA_DIR, 'Neon_Agnostic_col5pct.csv'), 'label': r'Neon Agnostic (5\% Col)', 'color': '#2ca02c', 'linestyle': ':'},
        {'filepath': os.path.join(DATA_DIR, 'Neon_Tight_col5pct.csv'), 'label': r'Neon Tight (5\% Col)', 'color': '#2ca02c', 'linestyle': '-.'},
    ]
    
    plot_percentage_collaterals_inset(pct_configs, y_max=2000)

Optional: compare this run with the precomputed results in `data/`, which were generated with `real_distribution_69` and used for the figures of the paper.

In [ ]:
## Optional reproducibility check (only meaningful for real_distribution_69).
## Rows are compared as parsed CSV, so different line endings (CRLF/LF) do not matter.
if DISTRIBUTION == "real_distribution_69" and os.path.isdir("data"):
    for name in sorted(f for f in os.listdir("data") if f.endswith(".csv")):
        new_file = os.path.join(DATA_DIR, name)
        if not os.path.exists(new_file):
            print(f"missing in this run: {name}")
            continue
        with open(os.path.join("data", name), newline='') as f_ref, open(new_file, newline='') as f_new:
            same = list(csv.reader(f_ref)) == list(csv.reader(f_new))
        print(f"{'identical' if same else 'DIFFERENT'}: {name}")
else:
    print("Skipped: data/ holds the results of real_distribution_69 only.")